# Module 8 | Class 2 -- MLflow Experiment Tracking

**Objective:** Learn how to track ML experiments (parameters, metrics, models) using MLflow so you can compare runs systematically instead of keeping notes in a spreadsheet.

We will:
1. Load and preprocess the Telco Churn dataset
2. Train 3 different models
3. Log each run to MLflow with parameters and metrics
4. Compare all runs in a single table

## 1. Setup

In [ ]:
!pip install -q mlflow scikit-learn pandas matplotlib

In [ ]:
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

RANDOM_STATE = 42

# Use local file-based tracking (no server needed)
mlflow.set_tracking_uri("file:///tmp/mlflow_runs")

print(f"MLflow version: {mlflow.__version__}")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")
print("Setup complete.")

## 2. Load and Preprocess Telco Churn Data

In [ ]:
# Load Telco Churn dataset
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)
print(f"Shape: {df.shape}")
print(f"Churn distribution:\n{df['Churn'].value_counts()}")
df.head()

In [ ]:
# Preprocess
df_proc = df.copy()

# Fix TotalCharges (has some spaces that should be NaN)
df_proc['TotalCharges'] = pd.to_numeric(df_proc['TotalCharges'], errors='coerce')
df_proc['TotalCharges'].fillna(df_proc['TotalCharges'].median(), inplace=True)

# Drop customerID
df_proc.drop('customerID', axis=1, inplace=True)

# Encode target
df_proc['Churn'] = (df_proc['Churn'] == 'Yes').astype(int)

# Encode categorical columns
cat_cols = df_proc.select_dtypes(include='object').columns.tolist()
le_dict = {}
for col in cat_cols:
    le = LabelEncoder()
    df_proc[col] = le.fit_transform(df_proc[col])
    le_dict[col] = le

print(f"Encoded {len(cat_cols)} categorical columns: {cat_cols}")
print(f"Final shape: {df_proc.shape}")

In [ ]:
# Split data
X = df_proc.drop('Churn', axis=1)
y = df_proc['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train: {X_train_scaled.shape} | Test: {X_test_scaled.shape}")

## 3. Define Training + Logging Function

This function trains a model and logs everything to MLflow: parameters, metrics, and the model artifact.

In [ ]:
def train_and_log(model, model_name, params, X_tr, X_te, y_tr, y_te):
    """
    Train a model, evaluate it, and log everything to MLflow.

    Args:
        model: sklearn estimator (already instantiated)
        model_name: string identifier for this run
        params: dict of hyperparameters to log
        X_tr, X_te, y_tr, y_te: train/test data
    """
    with mlflow.start_run(run_name=model_name):
        # Log parameters
        for key, value in params.items():
            mlflow.log_param(key, value)
        mlflow.log_param("model_type", model_name)

        # Train
        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_te)

        # Calculate metrics
        acc = accuracy_score(y_te, y_pred)
        prec = precision_score(y_te, y_pred)
        rec = recall_score(y_te, y_pred)
        f1 = f1_score(y_te, y_pred)

        # Log metrics
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("precision", prec)
        mlflow.log_metric("recall", rec)
        mlflow.log_metric("f1_score", f1)

        # Log model
        mlflow.sklearn.log_model(model, "model")

        print(f"[{model_name}] Acc={acc:.4f} | Prec={prec:.4f} | Rec={rec:.4f} | F1={f1:.4f}")

    return model

print("Training function ready.")

## 4. Train 3 Models and Log to MLflow

In [ ]:
# Set experiment name
mlflow.set_experiment("telco_churn_comparison")

print("Training 3 models...\n")

In [ ]:
# Model 1: Logistic Regression
m1 = train_and_log(
    model=LogisticRegression(max_iter=1000, C=1.0, random_state=RANDOM_STATE),
    model_name="LogisticRegression",
    params={"C": 1.0, "max_iter": 1000},
    X_tr=X_train_scaled, X_te=X_test_scaled,
    y_tr=y_train, y_te=y_test
)

In [ ]:
# Model 2: Random Forest (n_estimators=50)
m2 = train_and_log(
    model=RandomForestClassifier(n_estimators=50, max_depth=10, random_state=RANDOM_STATE),
    model_name="RandomForest_n50",
    params={"n_estimators": 50, "max_depth": 10},
    X_tr=X_train_scaled, X_te=X_test_scaled,
    y_tr=y_train, y_te=y_test
)

In [ ]:
# Model 3: Random Forest (n_estimators=200)
m3 = train_and_log(
    model=RandomForestClassifier(n_estimators=200, max_depth=15, random_state=RANDOM_STATE),
    model_name="RandomForest_n200",
    params={"n_estimators": 200, "max_depth": 15},
    X_tr=X_train_scaled, X_te=X_test_scaled,
    y_tr=y_train, y_te=y_test
)

## 5. Compare All Runs

MLflow stores every run. We can query them and build a comparison table.

In [ ]:
# Search all runs in our experiment
experiment = mlflow.get_experiment_by_name("telco_churn_comparison")
runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])

# Select relevant columns
cols = [
    'run_id',
    'params.model_type',
    'metrics.accuracy',
    'metrics.precision',
    'metrics.recall',
    'metrics.f1_score'
]
available_cols = [c for c in cols if c in runs.columns]

comparison = runs[available_cols].copy()
comparison.columns = [c.split('.')[-1] for c in available_cols]

# Sort by F1 score
comparison = comparison.sort_values('f1_score', ascending=False)

print("MLflow Experiment Comparison:")
print("=" * 70)
print(comparison.to_string(index=False))

In [ ]:
# Visualize comparison
metrics = ['accuracy', 'precision', 'recall', 'f1_score']
available_metrics = [m for m in metrics if m in comparison.columns]
model_names = comparison['model_type'].tolist()

x = np.arange(len(model_names))
width = 0.2

fig, ax = plt.subplots(figsize=(10, 5))
for i, metric in enumerate(available_metrics):
    values = comparison[metric].tolist()
    ax.bar(x + i * width, values, width, label=metric)

ax.set_ylabel('Score')
ax.set_title('Model Comparison (MLflow Tracked)')
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(model_names)
ax.legend()
ax.set_ylim(0.5, 1.0)
plt.tight_layout()
plt.show()

## 6. Key Takeaways

- `mlflow.log_param()` stores hyperparameters (what you configured)
- `mlflow.log_metric()` stores results (what you measured)
- `mlflow.sklearn.log_model()` saves the model artifact for later loading
- `mlflow.search_runs()` retrieves all runs for comparison
- In production, you would point `set_tracking_uri()` to a remote MLflow server so the whole team can see results

---
## TODO: Student Exercise

Add 2 more model variants to the experiment and log them.

**Requirements:**
1. Train at least 2 additional models. Ideas:
   - LogisticRegression with `C=0.01` or `C=10.0`
   - RandomForest with `n_estimators=100, max_depth=5`
   - `GradientBoostingClassifier` (import from `sklearn.ensemble`)
   - `SVC` with `probability=True` (import from `sklearn.svm`)
2. Use `train_and_log()` for each
3. After training, run `mlflow.search_runs()` again and show the updated comparison table
4. Which model has the best F1 score now?

In [ ]:
# TODO: Model 4
# train_and_log(
#     model=...,
#     model_name="...",
#     params={...},
#     X_tr=X_train_scaled, X_te=X_test_scaled,
#     y_tr=y_train, y_te=y_test
# )


In [ ]:
# TODO: Model 5


In [ ]:
# TODO: Show updated comparison table
# Rerun mlflow.search_runs() and display
